# G1 multi-tarefa — sessão isolada

Ambiente **próprio**, separado de qualquer notebook da Lift. O contrato de isolamento:

| o quê | onde |
|---|---|
| repo | `/kaggle/input/g1-multitask/` (dataset privado, read-only) |
| cópia gravável | `/kaggle/working/g1_mt/` |
| logs de treino | `/kaggle/working/g1_mt/logs/` |
| task registrada | `Mjlab-Multitask-Unitree-G1` |

Nada aqui toca `g1_training/` — o pacote `g1_multitask/` só importa de lá.

**Ordem das células, e o motivo de cada uma:**

1. ambiente — vê o que a Kaggle deu, ANTES de instalar nada
2. instalar — deps pinadas no que roda local
3. verificar — torch/CUDA sobreviveram ao pip? é aqui que se perde a GPU sem perceber
4. montar o repo — cópia gravável
5. simulação do currículo — segundos, sem física: prova os 60 destravamentos
6. **pré-voo em GPU — o item 0**: `dr.body_com_offset` corrompe a heap em CPU, e nunca foi testado em GPU
7. treino — 1000 iterações
8. **resume — o portão do plano**
9. relatório entre blocos

## Ajuste de expectativa

Com política do zero, **o currículo não destrava nada em 1000 iterações**. O portão é
0.90 de sucesso absoluto, e o sucesso do `parado` é sobreviver 20 s sem cair — coisa
que rede aleatória não faz. Você vai ver a EMA **caindo** de 0.5 (valor inicial) na
direção do sucesso real.

1000 iterações valida **encanamento**: não quebrou, throughput, pico de VRAM, e o
resume. Não valida currículo.


## 1. Ambiente, antes de instalar nada

In [ ]:
import subprocess, sys, os, pathlib
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
import torch
print(f"torch {torch.__version__}   cuda={torch.version.cuda}   "
      f"disponível={torch.cuda.is_available()}")
print(f"python {sys.version.split()[0]}")
TORCH_ANTES = torch.__version__
assert torch.cuda.is_available(), "sem GPU: Settings -> Accelerator -> GPU T4 x1"

## 2. Instalar

O mjlab declara a árvore inteira (inclusive `rsl-rl-lib==5.4.0` exato), então instalar
ele puxa o resto. `torch` fica de fora de propósito: a Kaggle já traz um build casado
com o CUDA da imagem, e deixar o pip trocá-lo é o jeito mais rápido de perder a GPU.

In [ ]:
REQ = None
for base in pathlib.Path("/kaggle/input").glob("*/"):
    c = list(base.rglob("g1_multitask/kaggle/requirements.txt"))
    if c:
        REQ = c[0]; break
assert REQ, "dataset não encontrado em /kaggle/input — subiu o zip como Dataset?"
print("requirements:", REQ)
!pip install -q --no-warn-conflicts -r {REQ}

## 3. Verificar — é aqui que se descobre a GPU perdida

Se o pip trocou o torch por um build sem CUDA, tudo abaixo roda em CPU sem reclamar e
a sessão vira 12 h de nada.

In [ ]:
import importlib
importlib.invalidate_caches()
import torch  # noqa
importlib.reload(torch)
print(f"torch {torch.__version__} (era {TORCH_ANTES})   cuda={torch.cuda.is_available()}")
assert torch.cuda.is_available(), (
    "o pip trocou o torch e a CUDA foi embora. Reinicie o kernel e instale com "
    "`--no-deps` os pacotes que puxaram torch.")
import mjlab, mujoco, warp, rsl_rl
print("mjlab", mjlab.__version__ if hasattr(mjlab, "__version__") else "ok",
      "| mujoco", mujoco.__version__, "| warp", warp.config.version)

## 4. Montar o repo numa cópia gravável

In [ ]:
import shutil
FONTE = REQ.parent.parent.parent          # .../g1_multitask/kaggle/req.txt -> raiz do repo
RAIZ = pathlib.Path("/kaggle/working/g1_mt")
if RAIZ.exists():
    shutil.rmtree(RAIZ)
shutil.copytree(FONTE, RAIZ, ignore=shutil.ignore_patterns(
    "__pycache__", "*.pyc", ".venv", "logs", "runs"))
os.chdir(RAIZ)
print("raiz:", RAIZ)
print("topo:", sorted(p.name for p in RAIZ.iterdir())[:12])

sys.path.insert(0, str(RAIZ))
import g1_multitask
from mjlab.tasks.registry import list_tasks
print("\ntask registrada:", g1_multitask.TASK_ID in list_tasks())
print("tasks visíveis:", [t for t in list_tasks() if "Unitree-G1" in t])

## 5. Simulação do currículo — segundos, sem física

Prova que a sequência de destravamentos do código é a do desenho: 60 no total, cadeia
de profundidade 9, cada eixo esgotado. Se isto falhar, não submeta treino.

In [ ]:
!python g1_multitask/sim_curriculo.py 2>&1 | tail -25

## 6. Pré-voo em GPU — **o item 0**

`dr.body_com_offset` **corrompe a heap no backend CPU do warp** (medido 30/07: core
dump, e derruba a task do próprio fabricante do mesmo jeito). Ele fica LIGADO no config
porque o treino roda em GPU, onde o caminho de kernel é outro — mas isso nunca foi
verificado. É agora.

Se esta célula derrubar o processo: `DR(base_com=False)` no config, e segue. Perde-se
±2.5 cm de randomização de CoM, não se perde a run.

In [ ]:
!python g1_multitask/preflight_gpu.py 4096

## 7. Treino — 1000 iterações

**Uma GPU só**, mesmo tendo duas: com dual T4 o `torchrunx` manda o stdout dos workers
para arquivo e as linhas `[CURRICULO]` desaparecem da saída da célula — justamente o
que você quer ver.

`num_envs` é **por rank**. Com uma GPU, 4096 é 4096.

In [ ]:
!python g1_multitask/train.py \
    --gpu-ids "[0]" \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 1000 \
    2>&1 | grep -vE "^Module |took .* ms" | tail -60

## 8. Resume — **o portão do plano**

Com a run fatiada em blocos de 2k–3k, `save`/`load` dispara de 10 a 15 vezes. Um bug
aqui perde o currículo em **silêncio**: o treino segue rodando, só volta pro nível 0.

Procure a linha:

```
[CURRICULO] retomado: N/60 eventos, M tarefas abertas, push nível K
```

In [ ]:
!python g1_multitask/train.py \
    --gpu-ids "[0]" \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 100 \
    --agent.resume True \
    2>&1 | grep -E "CURRICULO|Loading model|resume|Error|Traceback" | head -20

## 9. Relatório entre blocos

In [ ]:
!python g1_multitask/entre_blocos.py 2>&1 | head -80

## 10. TensorBoard (opcional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/g1_mt/logs

## O que olhar, e o que significa

| olhar | onde | bom sinal |
|---|---|---|
| não quebrou | célula 7 | zero NaN, zero crash, `base_com` sobreviveu |
| throughput | célula 6 | dimensiona os blocos de verdade |
| pico de VRAM | célula 6 | só a sessão em GPU revela |
| **resume** | célula 8 | a linha `[CURRICULO] retomado` |
| está aprendendo? | `Episode_Reward/track_linear_velocity` | subindo = locomoção pegou |
| régua viva? | `Episode_Metrics/sucesso` | saiu de 0 em alguma tarefa? |

### Se quiser ver o currículo destravar nesta sessão

Dá para forçar, baixando o limiar num config de teste:

```python
from dataclasses import replace
from g1_multitask.knobs import MultitaskKnobs
TESTE = MultitaskKnobs()
TESTE.curriculum.limiar_competencia = 0.30   # em vez de 0.90
```

Aí a Fase 0 anda e você vê a cascata de aberturas ao vivo. **Não deixe isso virar o
baseline:** mudar a definição de sucesso é Categoria C — joga o checkpoint no lixo.

### O que pode mudar entre blocos

| categoria | exemplo | custo |
|---|---|---|
| **A** | peso de reward, ligar/desligar gate, `ema_alpha` | **grátis**, retoma do checkpoint |
| **B** | enxertar canal novo na obs, reinicializar o crítico | warm-start |
| **C** | largura da obs (151), espaço de ação, **definição de sucesso** | do zero |

A Categoria A só é grátis porque o sucesso mora em `env.success_buf` como **fato
físico**: mexer em peso não move a régua e não invalida nenhuma EMA.
